# Этап 1: выгрузка результатов в CSV

Нажмите **Run All**, затем подставьте свой файл и запустите ещё раз.
Файлы появятся в папке `data/examples/export/` (она не попадает в git).

In [ ]:
from pathlib import Path

import pandas as pd
from ramancore.export import table_to_csv, to_csv
from ramancore.io import read_spectrum
from ramancore.library import load_library
from ramancore.matching import search
from ramancore.peaks import find_peaks, fit_peaks
from ramancore.preprocess import crop, subtract_baseline

out = Path("../data/examples/export")
out.mkdir(exist_ok=True)  # создать папку, если её ещё нет

In [ ]:
my_file = "../data/examples/synthetic_si.txt"  # ← подставьте свой файл
x, y, meta = read_spectrum(my_file, laser_nm=532)
name = Path(my_file).stem  # имя файла без расширения — для имён выгрузок

## 1. Спектр: исходный, фон и спектр без фона

Столбцы: `raman_shift_cm-1`, `intensity` (исходный), `baseline`, `corrected`.
В шапке-комментарии — файл, лазер, параметры фона и история обработки.

In [ ]:
xc, yc = crop(x, y)
corrected, baseline = subtract_baseline(xc, yc, method="asls", lam=1e5)
meta["history"].append("subtract_baseline: asls, lam=1e5")
path = to_csv(out / f"{name}_spectrum.csv", xc, yc, meta,
              params={"baseline_method": "asls", "lam": 1e5},
              extra={"baseline": baseline, "corrected": corrected})
print(path.read_text(encoding="utf-8")[:600])  # первые строки файла

## 2. Таблица пиков

Центр, FWHM, высота, площадь и их погрешности 1σ.

In [ ]:
table, fit = fit_peaks(xc, yc, find_peaks(xc, yc), shape="lorentz")
table_to_csv(out / f"{name}_peaks.csv", table, meta, params={"shape": "lorentz"})
table.round(2)

## 3. Результат поиска по базе

In [ ]:
lib = load_library("../data/library")
result = search(x, y, lib, top=10)
table_to_csv(out / f"{name}_search.csv", result, meta, params=lib["params"])
result.round(1)

## 4. Проверка: читается ли обратно

Спектр читает `read_spectrum` (строки `#` он пропускает), таблицы — `pd.read_csv(..., comment="#")`.

In [ ]:
x2, y2, _ = read_spectrum(out / f"{name}_spectrum.csv")
print("Спектр совпал точно:", (x2 == xc).all() and (y2 == yc).all())
pd.read_csv(out / f"{name}_peaks.csv", comment="#").round(2)